In [2]:
from pathlib import Path
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
import soundfile as sf

from remotezip import RemoteZip
from imageio_ffmpeg import get_ffmpeg_exe

In [3]:
AUDIO_DIR = Path(".")
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"

META_DIR = DATA_DIR / "metadata"

PRIMARY_MANIFEST_PATH = (
    META_DIR / "benchmark_2_audio_primary_manifest.csv"
)

MODEL_MANIFEST_PATH = (
    META_DIR / "benchmark_2_audio_model_manifest.csv"
)

AUDIO_CACHE_DIR = (
    DATA_DIR / "audio_cache" / "full"
)

AUDIO_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

(AUDIO_DIR / "models").mkdir(
    parents=True,
    exist_ok=True
)

(AUDIO_DIR / "results").mkdir(
    parents=True,
    exist_ok=True
)

In [4]:

MAMMALPS_URL = (
    "https://zenodo.org/records/15588220/"
    "files/mammalps_v1.zip"
)

VAL_FRACTION = 0.20
RANDOM_SEED = 42


# Context lengths
CONTEXT_OPTIONS_S = [
    0.96,   # YAMNet native frame
    1.0,    # gate seconds
    2.0,    # second adaptive stage
    3.0,    # BirdNET
]

FFMPEG = get_ffmpeg_exe()

print("Primary manifest:", PRIMARY_MANIFEST_PATH)
print("Audio cache:", AUDIO_CACHE_DIR)
print("FFmpeg:", FFMPEG)

Primary manifest: ..\data\metadata\benchmark_2_audio_primary_manifest.csv
Audio cache: ..\data\audio_cache\full
FFmpeg: C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe


In [5]:
manifest = pd.read_csv(
    PRIMARY_MANIFEST_PATH
)

print("Samples:", len(manifest))
print("Events:", manifest["event_id"].nunique())
print("Files:", manifest["file_id"].nunique())

Samples: 908
Events: 389
Files: 779


In [6]:
train_events = (
    manifest.loc[
        manifest["split"] == "train",
        ["event_id", "label_name", "site_id"]
    ]
    .drop_duplicates()
    .copy()
)

assert (
    train_events
    .groupby("event_id")["site_id"]
    .nunique()
    .max()
    == 1
)

rng = np.random.default_rng(
    RANDOM_SEED
)

val_event_ids = set()

for (label, site), group in train_events.groupby(
    ["label_name", "site_id"]
):

    event_ids = np.array(
        sorted(group["event_id"].unique())
    )

    rng.shuffle(event_ids)

    if len(event_ids) < 2:
        n_val = 0
    else:
        n_val = round(
            len(event_ids) * VAL_FRACTION
        )

        n_val = max(1, n_val)
        n_val = min(
            n_val,
            len(event_ids) - 1
        )

    val_event_ids.update(
        event_ids[:n_val]
    )

In [7]:
manifest["dataset_split"] = "train"

manifest.loc[
    manifest["event_id"].isin(val_event_ids),
    "dataset_split"
] = "val"

manifest.loc[
    manifest["split"] == "test",
    "dataset_split"
] = "test"

In [8]:
assert (
    manifest
    .groupby("event_id")["dataset_split"]
    .nunique()
    .max()
    == 1
)

assert (
    manifest.loc[
        manifest["split"] == "test",
        "dataset_split"
    ]
    == "test"
).all()

split_summary = (
    manifest
    .groupby(
        ["dataset_split", "label_name"]
    )
    .agg(
        samples=("sample_id", "count"),
        events=("event_id", "nunique"),
        files=("file_id", "nunique")
    )
)

split_summary

samples  events  files
dataset_split label_name                        
test          animal          123      57    117
              background       84      28     48
train         animal          436     203    425
              background      120      40     59
val           animal          115      51    111
              background       30      10     19

In [9]:
def context_summary(
    df,
    context_seconds
):
    valid = df[
        df["duration"] >= context_seconds
    ]

    return {
        "context_s": context_seconds,
        "samples": len(valid),
        "events": valid["event_id"].nunique(),
        "animal_events": valid.loc[
            valid["label_name"] == "animal",
            "event_id"
        ].nunique(),
        "background_events": valid.loc[
            valid["label_name"] == "background",
            "event_id"
        ].nunique(),
    }

In [10]:
context_table = pd.DataFrame(
    [
        context_summary(
            manifest,
            context_s
        )
        for context_s in CONTEXT_OPTIONS_S
    ]
)

context_table

,context_s,samples,events,animal_events,background_events
0,0.96,908,389,311,78
1,1.00,908,389,311,78
2,2.00,908,389,311,78
3,3.00,836,374,296,78


In [11]:
file_table = (
    manifest[
        [
            "file_id",
            "remote_video_path",
            "duration"
        ]
    ]
    .drop_duplicates("file_id")
    .copy()
)

assert (
    file_table
    .groupby("file_id")["remote_video_path"]
    .nunique()
    .max()
    == 1
)

file_table["audio_path"] = (
    file_table["file_id"]
    .apply(
        lambda x:
        AUDIO_CACHE_DIR / f"{x}.flac"
    )
)

print(
    "Unique audio files required:",
    len(file_table)
)

Unique audio files required: 779


In [12]:
audio_path_map = dict(
    zip(
        file_table["file_id"],
        file_table["audio_path"]
    )
)

manifest["audio_path"] = (
    manifest["file_id"]
    .map(audio_path_map)
    .astype(str)
)

In [13]:
def extract_audio_from_video(
    video_path,
    audio_path
):
    command = [
        FFMPEG,
        "-loglevel", "error",
        "-y",
        "-i", str(video_path),

        # First audio stream only
        "-map", "0:a:0",

        # No video
        "-vn",

        # All target models use mono audio
        "-ac", "1",

        # Lossless, preserve native sample rate
        "-c:a", "flac",

        str(audio_path),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr
        )

In [14]:
failures = []

with RemoteZip(MAMMALPS_URL) as z:

    archive_names = set(
        z.namelist()
    )

    missing_remote = (
        set(file_table["remote_video_path"])
        - archive_names
    )

    assert not missing_remote, (
        f"{len(missing_remote)} video paths "
        "were not found in the MammAlps archive."
    )

    total = len(file_table)

    with tempfile.TemporaryDirectory() as tmp:

        tmp_dir = Path(tmp)

        for count, (_, row) in enumerate(file_table.iterrows(), start=1):

            output_path = Path(
                row["audio_path"]
            )

            # Cache is persistent: skip files
            # successfully extracted before.
            if output_path.exists():
                continue

            remote_path = row[
                "remote_video_path"
            ]

            temp_video = (
                tmp_dir /
                f"{row['file_id']}.mp4"
            )

            try:

                # Stream MP4 from remote ZIP
                # instead of loading the entire
                # video into RAM.
                with z.open(remote_path) as src:
                    with open(
                        temp_video,
                        "wb"
                    ) as dst:

                        shutil.copyfileobj(
                            src,
                            dst,
                            length=1024 * 1024
                        )

                extract_audio_from_video(
                    temp_video,
                    output_path
                )

            except Exception as exc:

                failures.append({
                    "file_id":
                        row["file_id"],
                    "remote_video_path":
                        remote_path,
                    "error":
                        str(exc),
                })

            finally:

                if temp_video.exists():
                    temp_video.unlink()

            if count % 25 == 0:
                print(f"{count}/{total}")

25/779
50/779
75/779
175/779
200/779
225/779
250/779
275/779
300/779
325/779
400/779
425/779
450/779
475/779
500/779
525/779
550/779
575/779
600/779
675/779
700/779
725/779
750/779
775/779
825/779
850/779
875/779
900/779


In [15]:
audio_rows = []

for _, row in file_table.iterrows():

    audio_path = Path(
        row["audio_path"]
    )

    if not audio_path.exists():

        audio_rows.append({
            "file_id":
                row["file_id"],
            "audio_exists":
                False,
        })

        continue

    info = sf.info(
        audio_path
    )

    audio_rows.append({
        "file_id":
            row["file_id"],
        "audio_exists":
            True,
        "sample_rate":
            info.samplerate,
        "channels":
            info.channels,
        "frames":
            info.frames,
        "audio_duration_s":
            info.frames /
            info.samplerate,
    })

audio_index = pd.DataFrame(
    audio_rows
)

audio_index.head()

,file_id,audio_exists,sample_rate,channels,frames,audio_duration_s
0,S1_C1_E130_V0579,True,44100,1,132240,2.998639
1,S1_C1_E130_V0581,True,44100,1,132240,2.998639
2,S1_C1_E130_V0585,True,44100,1,132240,2.998639
3,S1_C2_E131_V0289,True,44100,1,132240,2.998639
4,S1_C2_E131_V0296,True,44100,1,264480,5.997279


In [16]:
print(
    "Missing audio:",
    (~audio_index["audio_exists"])
    .sum()
)

print("\nSample rates:")
print(
    audio_index["sample_rate"]
    .value_counts()
)

print("\nChannels:")
print(
    audio_index["channels"]
    .value_counts()
)

print("\nFailures:")
print(
    len(failures)
)

Missing audio: 0

Sample rates:
sample_rate
44100    779
Name: count, dtype: int64

Channels:
channels
1    779
Name: count, dtype: int64

Failures:
0


In [17]:
AUDIO_INDEX_PATH = (
    META_DIR /
    "benchmark_2_audio_cache_index.csv"
)

audio_index.to_csv(
    AUDIO_INDEX_PATH,
    index=False
)

manifest.to_csv(
    MODEL_MANIFEST_PATH,
    index=False
)

print("Saved:")
print(MODEL_MANIFEST_PATH)
print(AUDIO_INDEX_PATH)

Saved:
..\data\metadata\benchmark_2_audio_model_manifest.csv
..\data\metadata\benchmark_2_audio_cache_index.csv
